In [1]:
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import itertools

os.chdir("..")
from src.config import FIGURES_DIR, INTERIM_DATA_DIR

df = pd.read_csv(INTERIM_DATA_DIR / "ALL_PRESCRIPTION_DATA_FILTERED.csv")
df["Prescription Date"] = pd.to_datetime(df["Prescription Date"], format="%Y%m%d", errors="coerce")
df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")

2025-08-27 22:41:52.709 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: D:\Research\Project_COPD\COPD
C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_72040\1892021385.py:13: DtypeWarning: Columns (2,4,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INTERIM_DATA_DIR / "ALL_PRESCRIPTION_DATA_FILTERED.csv")


In [2]:
def plot_patient_variables_grid(
    dfx: pd.DataFrame,
    patient_id: int,
    out_dir: Path = Path(FIGURES_DIR / "pft_plots_all_variants"),
    plot_variable_order: List[str] = None,
    wanted_measure_indexes: List[str] = None,
    measure_index_colors: Dict[str, str] = None,
    marker: str = "o",
    linewidth: float = 1.6,
    plot_anomaly: bool = True,
    *,
    title_suffix: str = "",
) -> None:
    """
    Create one figure with vertical subplots (one per Variable).
    - Plots selected Measurements over time (per subplot).
    - Uses full YYYY-MM-DD dates on x-axis; only bottom subplot has x-label.
    - Overlays anomaly markers as hollow squares (no text labels).
    - Adds measurement legend (single row) and stacked anomaly legend lines.

    Expects anomaly columns if available:
      'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD', 'Outlier_Jump'
    The plot renders fine even if they are absent.
    """
    import itertools
    import matplotlib.dates as mdates
    from matplotlib.lines import Line2D

    wanted_measure_indexes = ['FEV1', 'FVC']

    if dfx.empty:
        return

    # ---- constants / basic prep ----
    DATE_COL  = "Prescription Date"
    VALUE_COL = "Result Numerical Value"
    VAR_COL   = "Variable"
    MEAS_COL  = "Measurement"

    dfx = dfx.copy()
    dfx[DATE_COL] = pd.to_datetime(dfx[DATE_COL], errors="coerce")
    dfx = dfx.sort_values(DATE_COL)

    if plot_variable_order is None:
        plot_variable_order = ["Meas", "%Pred", "%Chg.", "Post_Meas", "Post_%Pred", "Post_%Chg"]
    if wanted_measure_indexes is None:
        wanted_measure_indexes = sorted(dfx[MEAS_COL].dropna().unique().tolist())
    if measure_index_colors is None:
        base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
        measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}
    # Define marker mapping per Test
    MARKERS_BY_TEST = {"PFT": ".", "BD": "1", "COD": "^", None: "|"}
    MARKER_COLORS_BY_TEST = {"PFT": "red", "BD": "black", "COD": "green", None: "blue"}

    # anomaly colors (distinct from typical Matplotlib defaults)
    ANOM_COLORS = {
        "Missing": "#FFB300",  # amber
        "Range":   "#8B0000",  # dark red
        "MAD":     "#000000",  # black
        "Jump":    "#00BFA6",  # teal
    }
    have_anom_cols = all(
        col in dfx.columns
        for col in ["Anomaly_Missing", "Anomaly_Range", "Outlier_MAD", "Outlier_Jump"]
    )

    # ---- figure & axes ----
    nrows = len(plot_variable_order)
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=((16*2.3*nrows) / 9, 2.3*nrows), sharex=False)
    if nrows == 1:
        axes = [axes]

    handles_all, labels_all = [], []

    # ---- plotting per variable ----
    for ax, var in zip(axes, plot_variable_order):
        dft_var = dfx[dfx[VAR_COL] == var]
        ax.set_ylabel(var)
        ax.grid(True, linestyle="--", alpha=0.3)

        any_line = False
        for mi in wanted_measure_indexes:
            dft_sub  = dft_var[dft_var[MEAS_COL] == mi]
            if dft_sub.empty:
                continue

            # group by Test so each subgroup has its own marker
            for test_val, dft in dft_sub.groupby("Test"):
                h, = ax.plot(
                    dft[DATE_COL],
                    dft[VALUE_COL],
                    marker=MARKERS_BY_TEST.get(test_val, "|"),
                    markerfacecolor=MARKER_COLORS_BY_TEST.get(test_val, "blue"),
                    markeredgecolor=MARKER_COLORS_BY_TEST.get(test_val, "blue"),
                    linewidth=linewidth,
                    color=measure_index_colors.get(mi, "black"),
                    label=f"{test_val}-{mi}", zorder=2,
                )
                any_line = True
                if not any(lbl.get_label() == f"{test_val}-{mi}" for lbl in handles_all):
                    handles_all.append(h)
                    labels_all.append(f"{test_val}-{mi}")

                # anomaly overlays (squares) — only if anomaly cols exist
                if plot_anomaly and have_anom_cols:
                    anom_specs = [
                        ("MAD",     "Outlier_MAD"),
                        ("Jump",    "Outlier_Jump"),
                        ("Range",   "Anomaly_Range"),
                        ("Missing", "Anomaly_Missing"),
                    ]
                    for key, col in anom_specs:
                        if col in dft.columns and dft[col].any():
                            bad = dft[dft[col]]
                            ax.scatter(
                                bad[DATE_COL], bad[VALUE_COL],
                                s=80, marker="s",
                                facecolors="none", edgecolors=ANOM_COLORS[key],
                                linewidths=1.8, zorder=4
                            )

        if not any_line:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, alpha=0.6)

        # x-axis as full date per subplot
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        for tick in ax.get_xticklabels():
            tick.set_rotation(30)
            tick.set_ha("right")

    # x-label only on bottom subplot
    axes[-1].set_xlabel("Prescription Date")

    # title (with optional suffix)
    fig.suptitle(
        f"Patient {patient_id} — All Variables{(' — ' + title_suffix) if title_suffix else ''}",
        y=0.995, fontsize=14
    )

    # -------- LEGENDS (measurement row, then one anomaly per line) --------
    # 1) Measurement legend: single row
    if handles_all:
        fig.legend(
            handles_all, labels_all,
            loc="lower center", bbox_to_anchor=(0.5, 0.08),
            ncol=len(labels_all), fontsize=9, frameon=False,
            handlelength=2.0, handletextpad=0.6, columnspacing=1.2
        )

    # 2) One line per anomaly (stacked)
    anom_specs = [
        ("Missing Value: Value is zero.", ANOM_COLORS["Missing"]),
        ("Out of Valid Range: FEV1 & FVC (Meas/Post_Meas) 0.2-10.0; DLCO 0.3-50; FEV1/FVC 0.2-1.2; %Pred 0-200.", ANOM_COLORS["Range"]),
        ("MAD: |robust_z| > z_thresh (3.5)", ANOM_COLORS["MAD"]),
        ("Jump: Δ/month > thresholds (FEV1 0.30, FVC 0.40, DLCO 3.0, FEV1/FVC 0.08, DLCO/VA 0.60).", ANOM_COLORS["Jump"]),
    ]
    y0, dy = 0.07, 0.01   # starting y and spacing between lines
    for i, (lab, col) in enumerate(anom_specs):
        h = Line2D([0],[0], marker='s', linestyle='None', markersize=8,
                   markerfacecolor='none', markeredgecolor=col, label=lab)
        fig.legend(
            [h], [lab],
            loc="lower center", bbox_to_anchor=(0.5, y0 - i*dy),
            ncol=1, fontsize=9, frameon=False, handlelength=1.2
        )

    # room for 1 (measurements) + 4 (anomalies) lines
    fig.subplots_adjust(bottom=0.16, top=0.95, hspace=0.5)

    # ---- save ----
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"patient_{patient_id}_variables_grid.png"
    fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [6]:
patient_ids = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]

# for pid in patient_ids:
#     dfx = df[df["Patient Number"] == pid].copy()
#     dfx_flagged, title_tag = detect_anomalies(dfx)            # ← one function
#     plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

pid = 5665 #611957
dfx = df[df["Patient Number"] == pid].copy()

In [7]:
dc=['Patient Number', 'Prescription Date',  'Test', 'Measurement', 'Variable', 'Result Numerical Value']
dfx["Prescription Date"] = pd.to_datetime(dfx["Prescription Date"], errors="coerce")
dfx = dfx.sort_values(
    by=["Prescription Date", "Measurement", "Variable"]
)[dc].reset_index(drop=True)
dfx=dfx[dc]
dfx

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value
0,5665,2013-05-27,COD,DLCO,%Pred,123.00
1,5665,2013-05-27,COD,DLCO,Meas,20.00
2,5665,2013-05-27,COD,DLCO,Pred,16.20
3,5665,2013-05-27,BD,FEV1,%Chg.,9.00
4,5665,2013-05-27,BD,FEV1,%Pred,104.00
5,5665,2013-05-27,PFT,FEV1,%Pred,96.00
6,5665,2013-05-27,BD,FEV1,Meas,2.47
7,5665,2013-05-27,PFT,FEV1,Meas,2.27
8,5665,2013-05-27,PFT,FEV1,Pred,2.38
9,5665,2013-05-27,BD,FEV1/FVC,Meas,62.00


In [9]:
dfx[(dfx["Test"] == "COD") & (dfx["Measurement"] == "DLCO") & (dfx["Variable"] == "Meas")]

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value
1,5665,2013-05-27,COD,DLCO,Meas,20.0
19,5665,2013-11-20,COD,DLCO,Meas,17.3


In [10]:
dfx[dfx["Variable"] == "Meas"]

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value
1,5665,2013-05-27,COD,DLCO,Meas,20.00
6,5665,2013-05-27,BD,FEV1,Meas,2.47
7,5665,2013-05-27,PFT,FEV1,Meas,2.27
9,5665,2013-05-27,BD,FEV1/FVC,Meas,62.00
10,5665,2013-05-27,PFT,FEV1/FVC,Meas,60.00
15,5665,2013-05-27,BD,FVC,Meas,3.96
16,5665,2013-05-27,PFT,FVC,Meas,3.77
19,5665,2013-11-20,COD,DLCO,Meas,17.30
24,5665,2013-11-20,BD,FEV1,Meas,2.50
25,5665,2013-11-20,PFT,FEV1,Meas,2.27


In [11]:
date_col = "Prescription Date"
value_col = "Result Numerical Value"
measure_col = "Measurement"
variable_col = "Variable"
z_thresh = 3.5
jump_thresh_per_month = {"FEV1": 0.30, "FVC": 0.40, "DLCO": 3.0, "FEV1/FVC": 0.08, "DLCO/VA": 0.60,}
abs_vars   = {"Meas", "Post_Meas"}
perc_vars  = {"%Pred", "Post_%Pred"}

df2 = dfx.copy()

# Ensure numeric + datetime for calculations
v = pd.to_numeric(df2[value_col], errors="coerce")
dt = pd.to_datetime(df2[date_col], errors="coerce")
df2["_val_"] = v
df2["_date_"] = dt

# 1) Missing (as requested: treat 0 as missing; also NaN is missing)
df2["Anomaly_Missing"] = v.isna() | (v == 0)

In [12]:
dc = ['Patient Number', 'Test', 'Measurement',
       'Variable', '_date_', '_val_', 
       'Anomaly_Missing']
df2[dc]

,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing
0,5665,COD,DLCO,%Pred,2013-05-27,123.00,False
1,5665,COD,DLCO,Meas,2013-05-27,20.00,False
2,5665,COD,DLCO,Pred,2013-05-27,16.20,False
3,5665,BD,FEV1,%Chg.,2013-05-27,9.00,False
4,5665,BD,FEV1,%Pred,2013-05-27,104.00,False
5,5665,PFT,FEV1,%Pred,2013-05-27,96.00,False
6,5665,BD,FEV1,Meas,2013-05-27,2.47,False
7,5665,PFT,FEV1,Meas,2013-05-27,2.27,False
8,5665,PFT,FEV1,Pred,2013-05-27,2.38,False
9,5665,BD,FEV1/FVC,Meas,2013-05-27,62.00,False


In [13]:
rng_flag = pd.Series(False, index=df2.index)
def _violate(series_mask, low, high):
        if not series_mask.any(): 
            return pd.Series(False, index=df2.index)
        s = v.where(series_mask)
        return (s < low) | (s > high)

# Absolute measurements (Meas/Post_Meas)
m = df2[measure_col].astype(str)
var = df2[variable_col].astype(str)
mask_abs = var.isin(abs_vars)

rng_flag |= _violate(mask_abs & (m == "FEV1"), 0.2, 10.0)
rng_flag |= _violate(mask_abs & (m == "FVC"),  0.3, 10.0)
rng_flag |= _violate(mask_abs & (m == "DLCO"), 0.3, 50.0)

# Ratio FEV1/FVC (absolute)
rng_flag |= _violate(mask_abs & (m == "FEV1/FVC"), 0.2, 1.2)

# Percent predicted
mask_perc = var.isin(perc_vars)
rng_flag |= _violate(mask_perc, 0.0, 200.0)

df2["Anomaly_Range"] = rng_flag.fillna(False)

In [14]:
dc += ['Anomaly_Range']
for v in df2[variable_col].unique():
    print(f"Variable: {v}")
    display(df2.loc[df2[variable_col] == v][dc])

Variable: %Pred


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
0,5665,COD,DLCO,%Pred,2013-05-27,123.00,False,False
4,5665,BD,FEV1,%Pred,2013-05-27,104.00,False,False
5,5665,PFT,FEV1,%Pred,2013-05-27,96.00,False,False
13,5665,BD,FVC,%Pred,2013-05-27,107.00,False,False
14,5665,PFT,FVC,%Pred,2013-05-27,102.00,False,False
18,5665,COD,DLCO,%Pred,2013-11-20,106.00,False,False
22,5665,BD,FEV1,%Pred,2013-11-20,105.00,False,False
23,5665,PFT,FEV1,%Pred,2013-11-20,95.00,False,False
31,5665,BD,FVC,%Pred,2013-11-20,106.00,False,False
32,5665,PFT,FVC,%Pred,2013-11-20,102.00,False,False


Variable: Meas


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
1,5665,COD,DLCO,Meas,2013-05-27,20.00,False,False
6,5665,BD,FEV1,Meas,2013-05-27,2.47,False,False
7,5665,PFT,FEV1,Meas,2013-05-27,2.27,False,False
9,5665,BD,FEV1/FVC,Meas,2013-05-27,62.00,False,True
10,5665,PFT,FEV1/FVC,Meas,2013-05-27,60.00,False,True
15,5665,BD,FVC,Meas,2013-05-27,3.96,False,False
16,5665,PFT,FVC,Meas,2013-05-27,3.77,False,False
19,5665,COD,DLCO,Meas,2013-11-20,17.30,False,False
24,5665,BD,FEV1,Meas,2013-11-20,2.50,False,False
25,5665,PFT,FEV1,Meas,2013-11-20,2.27,False,False


Variable: Pred


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
2,5665,COD,DLCO,Pred,2013-05-27,16.20,False,False
8,5665,PFT,FEV1,Pred,2013-05-27,2.38,False,False
11,5665,PFT,FEV1/FVC,Pred,2013-05-27,67.00,False,False
17,5665,PFT,FVC,Pred,2013-05-27,3.70,False,False
20,5665,COD,DLCO,Pred,2013-11-20,16.20,False,False
26,5665,PFT,FEV1,Pred,2013-11-20,2.38,False,False
29,5665,PFT,FEV1/FVC,Pred,2013-11-20,67.00,False,False
35,5665,PFT,FVC,Pred,2013-11-20,3.70,False,False
38,5665,PFT,FEV1,Pred,2021-07-14,2.10,False,False
41,5665,PFT,FEV1/FVC,Pred,2021-07-14,0.00,True,False


Variable: %Chg.


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
3,5665,BD,FEV1,%Chg.,2013-05-27,9.0,False,False
12,5665,BD,FVC,%Chg.,2013-05-27,5.0,False,False
21,5665,BD,FEV1,%Chg.,2013-11-20,10.0,False,False
30,5665,BD,FVC,%Chg.,2013-11-20,4.0,False,False


In [15]:
# 3) Outliers (MAD + Jump) per measurement series
out_mad  = pd.Series(False, index=df2.index)
out_jump = pd.Series(False, index=df2.index)

In [50]:
df2.groupby(measure_col, dropna=False)

In [16]:
for mi, g in df2.groupby(measure_col, dropna=False):
    print(f"Measurement: {mi} (n={len(g)})")
    display(g[dc])

Measurement: DLCO (n=6)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
0,5665,COD,DLCO,%Pred,2013-05-27,123.0,False,False
1,5665,COD,DLCO,Meas,2013-05-27,20.0,False,False
2,5665,COD,DLCO,Pred,2013-05-27,16.2,False,False
18,5665,COD,DLCO,%Pred,2013-11-20,106.0,False,False
19,5665,COD,DLCO,Meas,2013-11-20,17.3,False,False
20,5665,COD,DLCO,Pred,2013-11-20,16.2,False,False


Measurement: FEV1 (n=18)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
3,5665,BD,FEV1,%Chg.,2013-05-27,9.00,False,False
4,5665,BD,FEV1,%Pred,2013-05-27,104.00,False,False
5,5665,PFT,FEV1,%Pred,2013-05-27,96.00,False,False
6,5665,BD,FEV1,Meas,2013-05-27,2.47,False,False
7,5665,PFT,FEV1,Meas,2013-05-27,2.27,False,False
8,5665,PFT,FEV1,Pred,2013-05-27,2.38,False,False
21,5665,BD,FEV1,%Chg.,2013-11-20,10.00,False,False
22,5665,BD,FEV1,%Pred,2013-11-20,105.00,False,False
23,5665,PFT,FEV1,%Pred,2013-11-20,95.00,False,False
24,5665,BD,FEV1,Meas,2013-11-20,2.50,False,False


Measurement: FEV1/FVC (n=12)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
9,5665,BD,FEV1/FVC,Meas,2013-05-27,62.00,False,True
10,5665,PFT,FEV1/FVC,Meas,2013-05-27,60.00,False,True
11,5665,PFT,FEV1/FVC,Pred,2013-05-27,67.00,False,False
27,5665,BD,FEV1/FVC,Meas,2013-11-20,64.00,False,True
28,5665,PFT,FEV1/FVC,Meas,2013-11-20,60.00,False,True
29,5665,PFT,FEV1/FVC,Pred,2013-11-20,67.00,False,False
39,5665,PFT,FEV1/FVC,%Pred,2021-07-14,0.00,True,False
40,5665,PFT,FEV1/FVC,Meas,2021-07-14,58.51,False,True
41,5665,PFT,FEV1/FVC,Pred,2021-07-14,0.00,True,False
48,5665,PFT,FEV1/FVC,%Pred,2023-01-10,0.00,True,False


Measurement: FVC (n=18)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
12,5665,BD,FVC,%Chg.,2013-05-27,5.00,False,False
13,5665,BD,FVC,%Pred,2013-05-27,107.00,False,False
14,5665,PFT,FVC,%Pred,2013-05-27,102.00,False,False
15,5665,BD,FVC,Meas,2013-05-27,3.96,False,False
16,5665,PFT,FVC,Meas,2013-05-27,3.77,False,False
17,5665,PFT,FVC,Pred,2013-05-27,3.70,False,False
30,5665,BD,FVC,%Chg.,2013-11-20,4.00,False,False
31,5665,BD,FVC,%Pred,2013-11-20,106.00,False,False
32,5665,PFT,FVC,%Pred,2013-11-20,102.00,False,False
33,5665,BD,FVC,Meas,2013-11-20,3.94,False,False


In [20]:
test_col: str = "Test"
for tmv, g in df2.groupby([test_col, measure_col, variable_col], dropna=False):
    if 'FEV1' in tmv and 'Meas' in tmv:
        print(f"Measurement: {tmv},  (n={len(g)})")
        display(g[dc])

Measurement: ('BD', 'FEV1', 'Meas'),  (n=2)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
6,5665,BD,FEV1,Meas,2013-05-27,2.47,False,False
24,5665,BD,FEV1,Meas,2013-11-20,2.50,False,False


Measurement: ('PFT', 'FEV1', 'Meas'),  (n=4)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
7,5665,PFT,FEV1,Meas,2013-05-27,2.27,False,False
25,5665,PFT,FEV1,Meas,2013-11-20,2.27,False,False
37,5665,PFT,FEV1,Meas,2021-07-14,1.77,False,False
46,5665,PFT,FEV1,Meas,2023-01-10,0.00,True,True


In [19]:
for mi, g in df2.groupby([measure_col, variable_col], dropna=False):
    if 'FEV1' in mi and 'Meas' in mi:
        print(f"Measurement: {mi},  (n={len(g)})")
        display(g[dc])

Measurement: ('FEV1', 'Meas'),  (n=6)


,Patient Number,Test,Measurement,Variable,_date_,_val_,Anomaly_Missing,Anomaly_Range
6,5665,BD,FEV1,Meas,2013-05-27,2.47,False,False
7,5665,PFT,FEV1,Meas,2013-05-27,2.27,False,False
24,5665,BD,FEV1,Meas,2013-11-20,2.50,False,False
25,5665,PFT,FEV1,Meas,2013-11-20,2.27,False,False
37,5665,PFT,FEV1,Meas,2021-07-14,1.77,False,False
46,5665,PFT,FEV1,Meas,2023-01-10,0.00,True,True


In [21]:
group = df2.groupby([test_col, measure_col, variable_col], dropna=False)
it = iter(group)

In [34]:
tmv, g = next(it)
print(tmv)

('PFT', 'FEV1', 'Meas')


In [35]:
display(g)

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,_val_,_date_,Anomaly_Missing,Anomaly_Range
7,5665,2013-05-27,PFT,FEV1,Meas,2.27,2.27,2013-05-27,False,False
25,5665,2013-11-20,PFT,FEV1,Meas,2.27,2.27,2013-11-20,False,False
37,5665,2021-07-14,PFT,FEV1,Meas,1.77,1.77,2021-07-14,False,False
46,5665,2023-01-10,PFT,FEV1,Meas,0.00,0.00,2023-01-10,True,True


In [36]:
g = g.sort_values("_date_")

In [37]:
display(g)

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,_val_,_date_,Anomaly_Missing,Anomaly_Range
7,5665,2013-05-27,PFT,FEV1,Meas,2.27,2.27,2013-05-27,False,False
25,5665,2013-11-20,PFT,FEV1,Meas,2.27,2.27,2013-11-20,False,False
37,5665,2021-07-14,PFT,FEV1,Meas,1.77,1.77,2021-07-14,False,False
46,5665,2023-01-10,PFT,FEV1,Meas,0.00,0.00,2023-01-10,True,True


In [100]:
g


,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,_val_,_date_,Anomaly_Missing,Anomaly_Range
6,5665,2013-05-27,Bronchodilator,FEV1,Meas,2.47,2.47,2013-05-27,False,False
7,5665,2013-05-27,PFT,FEV1,Meas,2.27,2.27,2013-05-27,False,False
24,5665,2013-11-20,Bronchodilator,FEV1,Meas,2.50,2.50,2013-11-20,False,False
25,5665,2013-11-20,PFT,FEV1,Meas,2.27,2.27,2013-11-20,False,False
37,5665,2021-07-14,PFT,FEV1,Meas,1.77,1.77,2021-07-14,False,False
46,5665,2023-01-10,PFT,FEV1,Meas,0.00,0.00,2023-01-10,True,True


In [38]:
g = g[g["Anomaly_Missing"]==False]
display(g)

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,_val_,_date_,Anomaly_Missing,Anomaly_Range
7,5665,2013-05-27,PFT,FEV1,Meas,2.27,2.27,2013-05-27,False,False
25,5665,2013-11-20,PFT,FEV1,Meas,2.27,2.27,2013-11-20,False,False
37,5665,2021-07-14,PFT,FEV1,Meas,1.77,1.77,2021-07-14,False,False


In [39]:
vv = g["_val_"]
dd = g["_date_"]

In [40]:
vv

7     2.27
25    2.27
37    1.77
Name: _val_, dtype: float64

In [41]:
med = vv.median()
med

2.27

In [42]:
np.abs(vv - med)

7     0.0
25    0.0
37    0.5
Name: _val_, dtype: float64

In [47]:
np.median(np.abs(vv - med))

0.0

In [43]:
mad = float(np.median(np.abs(vv - med))) if len(vv) else 0.0
mad

0.0

In [48]:
mad = mad if mad > 0 else 1e-9

In [49]:
0.5-2.27

-1.77

In [50]:
0.6745 *-1.77

-1.193865

In [51]:
-1.193865/ mad

-1193864999.9999998

In [52]:
mad = mad if mad > 0 else 1e-9
robust_z = 0.6745 * (vv - med) / mad
robust_z

7             0.0
25            0.0
37   -337250000.0
Name: _val_, dtype: float64

In [53]:
robust_z.abs() > 0.1

7     False
25    False
37     True
Name: _val_, dtype: bool

In [54]:
out_mad.loc[g.index] = robust_z.abs() > 0.1
out_mad

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37     True
38    False
39    False
40    False
41    False
42    False
43    False
44    False
45    False
46    False
47    False
48    False
49    False
50    False
51    False
52    False
53    False
dtype: bool

In [ ]:
# Jump per month

In [118]:
dv = vv.diff().abs()
dv

6      NaN
7     0.20
24    0.23
25    0.23
37    0.50
Name: _val_, dtype: float64

In [123]:
dd

6    2013-05-27
7    2013-05-27
24   2013-11-20
25   2013-11-20
37   2021-07-14
Name: _date_, dtype: datetime64[ns]

In [127]:
dd.diff()

6          NaT
7       0 days
24    177 days
25      0 days
37   2793 days
Name: _date_, dtype: timedelta64[ns]

In [128]:
dt_days = dd.diff().dt.days
dt_days

6        NaN
7        0.0
24     177.0
25       0.0
37    2793.0
Name: _date_, dtype: float64

In [146]:
dt_days = dt_days.where(dt_days > 1, 30)
dt_days

6       30.0
7       30.0
24     177.0
25      30.0
37    2793.0
Name: _date_, dtype: float64

In [147]:
months = dt_days / 30.0
months

6      1.0
7      1.0
24     5.9
25     1.0
37    93.1
Name: _date_, dtype: float64

In [148]:
thr = jump_thresh_per_month.get(str(mi[0]), np.inf)
thr

0.3

In [138]:
dv

6      NaN
7     0.20
24    0.23
25    0.23
37    0.50
Name: _val_, dtype: float64

In [140]:
months

6      0.033333
7      0.033333
24     5.900000
25     0.033333
37    93.100000
Name: _date_, dtype: float64

In [149]:
dv / months

6          NaN
7     0.200000
24    0.038983
25    0.230000
37    0.005371
dtype: float64

In [135]:
jump_thresh_per_month

{'FEV1': 0.3, 'FVC': 0.4, 'DLCO': 3.0, 'FEV1/FVC': 0.08, 'DLCO/VA': 0.6}

In [136]:
mi

('FEV1', 'Meas')

In [133]:
str(mi)

"('FEV1', 'Meas')"

In [141]:
(dv / months) > thr

6     False
7      True
24    False
25     True
37    False
dtype: bool

In [144]:
dt_days = dt_days.where(dt_days > 0, 30) 

6      1.0
7      1.0
24     5.9
25     1.0
37    93.1
Name: _date_, dtype: float64

In [142]:
display(g)

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,_val_,_date_,Anomaly_Missing,Anomaly_Range
6,5665,2013-05-27,Bronchodilator,FEV1,Meas,2.47,2.47,2013-05-27,False,False
7,5665,2013-05-27,PFT,FEV1,Meas,2.27,2.27,2013-05-27,False,False
24,5665,2013-11-20,Bronchodilator,FEV1,Meas,2.50,2.50,2013-11-20,False,False
25,5665,2013-11-20,PFT,FEV1,Meas,2.27,2.27,2013-11-20,False,False
37,5665,2021-07-14,PFT,FEV1,Meas,1.77,1.77,2021-07-14,False,False


In [150]:
# 3) Outliers (MAD + Jump) per measurement series
out_mad  = pd.Series(False, index=df2.index)
out_jump = pd.Series(False, index=df2.index)

#!  For a measurement currently considering all variables; we need for each variable

for mi, g in df2.groupby([measure_col, variable_col], dropna=False): #df2.groupby(measure_col, dropna=False):
    g = g[g["Anomaly_Missing"]==False]
    g = g.sort_values("_date_")
    vv = g["_val_"]
    dd = g["_date_"]

    # MAD
    med = vv.median()
    mad = float(np.median(np.abs(vv - med))) if len(vv) else 0.0
    mad = mad if mad > 0 else 1e-9
    robust_z = 0.6745 * (vv - med) / mad
    out_mad.loc[g.index] = robust_z.abs() > z_thresh

    # Jump per month
    dv = vv.diff().abs()
    dt_days = dd.diff().dt.days
    dt_days = dt_days.where(dt_days > 0, 1)  # avoid 0/NaN/<=0
    months = dt_days / 30.0
    # months = months.where(dt_days >= 14, 1.0) # ignore jumps if dt_days < 14
    thr = jump_thresh_per_month.get(str(mi[0]), np.inf) #jump_thresh_per_month.get(str(mi), np.inf)
    out_jump.loc[g.index] = (dv / months) > thr

df2["Outlier_MAD"]  = out_mad.fillna(False)
df2["Outlier_Jump"] = out_jump.fillna(False)
df2["Outlier"]      = df2["Outlier_MAD"] | df2["Outlier_Jump"]

In [153]:
df2[variable_col]

0     %Pred
1      Meas
2      Pred
3     %Chg.
4     %Pred
5     %Pred
6      Meas
7      Meas
8      Pred
9      Meas
10     Meas
11     Pred
12    %Chg.
13    %Pred
14    %Pred
15     Meas
16     Meas
17     Pred
18    %Pred
19     Meas
20     Pred
21    %Chg.
22    %Pred
23    %Pred
24     Meas
25     Meas
26     Pred
27     Meas
28     Meas
29     Pred
30    %Chg.
31    %Pred
32    %Pred
33     Meas
34     Meas
35     Pred
36    %Pred
37     Meas
38     Pred
39    %Pred
40     Meas
41     Pred
42    %Pred
43     Meas
44     Pred
45    %Pred
46     Meas
47     Pred
48    %Pred
49     Meas
50     Pred
51    %Pred
52     Meas
53     Pred
Name: Variable, dtype: object

In [157]:
df.demo = df2[(df2[measure_col] == 'FEV1') & (df2[variable_col] == 'Meas')].copy()

C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_56832\2214692293.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df.demo = df2[(df2[measure_col] == 'FEV1') & (df2[variable_col] == 'Meas')].copy()


In [158]:
df.demo.columns

Index(['Patient Number', 'Prescription Date', 'Test', 'Measurement',
       'Variable', 'Result Numerical Value', '_val_', '_date_',
       'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD', 'Outlier_Jump',
       'Outlier'],
      dtype='object')

In [161]:
df.demo[['Patient Number', 'Prescription Date', 'Test', 'Measurement',
       'Variable', 'Result Numerical Value', 'Outlier_Jump',
       'Outlier']]

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,Outlier_Jump,Outlier
6,5665,2013-05-27,Bronchodilator,FEV1,Meas,2.47,False,False
7,5665,2013-05-27,PFT,FEV1,Meas,2.27,True,True
24,5665,2013-11-20,Bronchodilator,FEV1,Meas,2.50,False,False
25,5665,2013-11-20,PFT,FEV1,Meas,2.27,True,True
37,5665,2021-07-14,PFT,FEV1,Meas,1.77,False,False
46,5665,2023-01-10,PFT,FEV1,Meas,0.00,False,False


In [163]:
# Group by 'Patient Number' and filter patients with at least 3 years between first and last prescription date
patient_groups = df.groupby('Patient Number')
def has_min_3_years(g):
    dates = pd.to_datetime(g["Prescription Date"], errors="coerce")
    if dates.isnull().all():
        return False
    return (dates.max() - dates.min()).days >= 3 * 365

patients_3yr = [pid for pid, g in patient_groups if has_min_3_years(g)]
df_3yr = df[df['Patient Number'].isin(patients_3yr)].copy()

In [164]:
df_3yr.shape

(291417, 19)

In [166]:
df.columns

Index(['Prescription Code', 'Patient Number', 'Gender', 'Date of Birth',
       'Visit Type', 'Treatment Date', 'Prescription Name',
       'Prescription Date', 'Implementation Date', 'Result item name',
       'Result Numerical Value', 'Laboratory', 'Implementation laboratory',
       'Region', 'Result Value', 'Pacs Number', 'Variable', 'Measurement',
       'Test'],
      dtype='object')

In [199]:
df_demo = df.copy()

In [169]:
df_demo.columns

Index(['Prescription Code', 'Patient Number', 'Gender', 'Date of Birth',
       'Visit Type', 'Treatment Date', 'Prescription Name',
       'Prescription Date', 'Implementation Date', 'Result item name',
       'Result Numerical Value', 'Laboratory', 'Implementation laboratory',
       'Region', 'Result Value', 'Pacs Number', 'Variable', 'Measurement',
       'Test'],
      dtype='object')

array(['Bronchodilator Test', 'PFT with Flow-Volume Curve(기본폐기능검사)',
       'CO Diffusing Capacity Measurement', 'Bronchodilator Test ',
       'PFT with Flow-Volume Curve(기본폐기능검사) ',
       'CO Diffusing Capacity Measurement ',
       '[임상] PFT with Flow-Volume Curve(기본폐기능검사)',
       '[응급] PFT with Flow-Volume Curve(기본폐기능검사)',
       '[협진]Bronchodilator Test',
       '[협진]PFT with Flow-Volume Curve(기본폐기능검사)',
       '[협진]CO Diffusing Capacity Measurement',
       '[임상] Bronchodilator Test', 'PFT without Flow-Volume Curve[수술전검사]'],
      dtype=object)

In [180]:
print(df_demo["Prescription Name"].nunique())
for t in df_demo["Prescription Name"].unique():
    print(t)

13
Bronchodilator Test
PFT with Flow-Volume Curve(기본폐기능검사)
CO Diffusing Capacity Measurement
Bronchodilator Test 
PFT with Flow-Volume Curve(기본폐기능검사) 
CO Diffusing Capacity Measurement 
[임상] PFT with Flow-Volume Curve(기본폐기능검사)
[응급] PFT with Flow-Volume Curve(기본폐기능검사)
[협진]Bronchodilator Test
[협진]PFT with Flow-Volume Curve(기본폐기능검사)
[협진]CO Diffusing Capacity Measurement
[임상] Bronchodilator Test
PFT without Flow-Volume Curve[수술전검사]


In [181]:
print(df_demo["Test"].nunique())
for t in df_demo["Test"].unique():
    print(t)

15
Bronchodilator Test
PFT with Flow-Volume Curve(기본폐기능검사)
CO Diffusing Capacity Measurement
Bronchodilator Test 
PFT with Flow-Volume Curve(기본폐기능검사) 
CO Diffusing Capacity Measurement 
[임상] PFT with Flow-Volume Curve(기본폐기능검사)
[응급] PFT with Flow-Volume Curve(기본폐기능검사)
[협진]Bronchodilator Test
[협진]PFT with Flow-Volume Curve(기본폐기능검사)
[협진]CO Diffusing Capacity Measurement
[임상] Bronchodilator Test
PFT
Bronchodilator
CO Diffusing


In [186]:
pns = df_demo["Prescription Name"].unique()
ts = df_demo["Test"].unique()

In [187]:
for t in ts:
    if t not in pns:
        print(t)

PFT
Bronchodilator
CO Diffusing


In [200]:
df_demo = df_demo.rename(columns={"Test": "Temp_Test"})

In [189]:
df_demo.columns

Index(['Prescription Code', 'Patient Number', 'Gender', 'Date of Birth',
       'Visit Type', 'Treatment Date', 'Prescription Name',
       'Prescription Date', 'Implementation Date', 'Result item name',
       'Result Numerical Value', 'Laboratory', 'Implementation laboratory',
       'Region', 'Result Value', 'Pacs Number', 'Variable', 'Measurement',
       'Temp_Test'],
      dtype='object')

In [204]:
df_demo.head()

,Prescription Code,Patient Number,Gender,Date of Birth,Visit Type,Treatment Date,Prescription Name,Prescription Date,Implementation Date,Result item name,Result Numerical Value,Laboratory,Implementation laboratory,Region,Result Value,Pacs Number,Variable,Measurement,Temp_Test
0,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Meas,2.26,NaN,NaN,NaN,NaN,NaN,Meas,FVC,Bronchodilator Test
1,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_%Pred,75.00,NaN,NaN,NaN,NaN,NaN,%Pred,FVC,Bronchodilator Test
2,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Post_Meas,2.35,NaN,NaN,NaN,NaN,NaN,Post_Meas,FVC,Bronchodilator Test
3,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Post_%Pred,79.00,NaN,NaN,NaN,NaN,NaN,Post_%Pred,FVC,Bronchodilator Test
4,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Post_%Chg,4.00,NaN,NaN,NaN,NaN,NaN,Post_%Chg,FVC,Bronchodilator Test


In [ ]:
def map_test_row(row):
    temp_test = row["Temp_Test"]
    presc_name = row["Prescription Name"]
    # Mapping logic based on Temp_Test
    if "CO Diffusing" in temp_test or "CO Diffusing" in presc_name:
        return "COD"
    elif "Bronchodilator" in temp_test or "Bronchodilator" in presc_name:
        return "BD"
    elif "PFT" in temp_test or "PFT" in presc_name:
        return "PFT"
    else:
        return None

# Apply mapping
df_demo["Test"] = df_demo.apply(map_test_row, axis=1)

# Find mismatches between Temp_Test and Prescription Name mapping
def get_test_from_string(s):
    if "CO Diffusing" in s:
        return "COD"
    elif "Bronchodilator" in s:
        return "BD"
    elif "PFT" in s:
        return "PFT"
    else:
        return None

temp_test_mapped = df_demo["Temp_Test"].apply(get_test_from_string)
presc_name_mapped = df_demo["Prescription Name"].apply(get_test_from_string)

mismatch_mask = temp_test_mapped != presc_name_mapped
mismatches = df_demo.loc[mismatch_mask, ["Temp_Test", "Prescription Name"]].copy()
mismatches["Temp_Test_Mapped"] = temp_test_mapped[mismatch_mask]
mismatches["Prescription_Name_Mapped"] = presc_name_mapped[mismatch_mask]

if not mismatches.empty:
    mismatches.to_csv("test_prescription_name_mismatches.csv", index=False)

In [201]:
# Find mismatches between Temp_Test and Prescription Name mapping
def get_test_from_string(s):
    if "CO Diffusing" in s:
        return "COD"
    elif "Bronchodilator" in s:
        return "BD"
    elif "PFT" in s:
        return "PFT"
    else:
        return None

temp_test_mapped = df_demo["Temp_Test"].apply(get_test_from_string)
presc_name_mapped = df_demo["Prescription Name"].apply(get_test_from_string)

mismatch_mask = temp_test_mapped != presc_name_mapped
mismatches = df_demo.loc[mismatch_mask, ["Temp_Test", "Prescription Name"]].copy()
mismatches["Temp_Test_Mapped"] = temp_test_mapped[mismatch_mask]
mismatches["Prescription_Name_Mapped"] = presc_name_mapped[mismatch_mask]

if not mismatches.empty:
    mismatches.to_csv("test_prescription_name_mismatches.csv", index=False)

In [203]:
mismatch_mask.sum()

0

In [ ]:
mismatches

In [193]:
df_demo["Test"].value_counts()

Test
PFT    445186
BD     282981
COD     65213
Name: count, dtype: int64

In [195]:
df_demo["Temp_Test"].value_counts()

Temp_Test
PFT                                         377000
Bronchodilator                              245191
PFT with Flow-Volume Curve(기본폐기능검사)          61222
CO Diffusing                                 54575
Bronchodilator Test                          32690
CO Diffusing Capacity Measurement             8799
PFT with Flow-Volume Curve(기본폐기능검사)           6795
Bronchodilator Test                           5058
CO Diffusing Capacity Measurement             1833
[임상] PFT with Flow-Volume Curve(기본폐기능검사)        73
[협진]PFT with Flow-Volume Curve(기본폐기능검사)         68
[협진]Bronchodilator Test                         36
[응급] PFT with Flow-Volume Curve(기본폐기능검사)        28
[협진]CO Diffusing Capacity Measurement            6
[임상] Bronchodilator Test                         6
Name: count, dtype: int64

In [196]:
df_demo["Test"].isna().any()

False

In [198]:
df_demo["Temp_Test"].isna().any()

False

In [ ]:
df_demo = df_demo.drop(columns=["Temp_Test"])

In [205]:
df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))

C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_56832\3529406939.py:1: DtypeWarning: Columns (2,4,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))


In [207]:
df["Test"].unique()

array(['BD', 'PFT', 'COD'], dtype=object)

In [ ]:
if df["Test"].isna().any():
        print("Some Temp_Test values could not be mapped to Test.")

In [210]:
print(f"Unique Tests: {df['Test'].unique()} ({df['Test'].nunique()})")

Unique Tests: ['BD' 'PFT' 'COD'] (3)


In [213]:
l = ['BD' 'PFT' 'COD']

In [215]:
df[df["Test"].isin(l)]["Test"]

Series([], Name: Test, dtype: object)

In [ ]:
df[df["Test"] not in l]["Test"]

In [216]:
df["Test"].isna().any()

False

In [218]:
from src.config import RAW_DATA_DIR, INTERIM_DATA_DIR
from loguru import logger

In [224]:
df_path = INTERIM_DATA_DIR / "ALL_PRESCRIPTION_DATA_TMV.csv"
df_path = Path(df_path)

desired_items = ["FVC", "FEV1", "DLCO", "FEV1/FVC"]  # sensible default

# read as text to avoid dtype warnings / keep leading zeros
df_org = pd.read_csv(df_path, dtype=str, encoding="utf-8-sig", low_memory=False)
df= df_org.copy()

logger.info(f"Unique patients before filtering: {df['Patient Number'].nunique()}")

# Filter patients with at least 3 years between first and last prescription date
patient_groups = df.groupby('Patient Number')
def has_min_3_years(g):
    dates = pd.to_datetime(g["Prescription Date"], errors="coerce")
    if dates.isnull().all():
        return False
    return (dates.max() - dates.min()).days >= 3 * 365
patients_3yr = [pid for pid, g in patient_groups if has_min_3_years(g)]
df = df[df['Patient Number'].isin(patients_3yr)]

logger.info(f"Unique patients after filtering patients < 3 years: {df['Patient Number'].nunique()}")

# Filter by Tests
df = df.rename(columns={"Test": "Temp_Test"})
def map_test(val):
    if "CO Diffusing" in val:
        return "COD"
    elif "Bronchodilator" in val:
        return "BD"
    elif "PFT" in val:
        return "PFT"
    else:
        return None
    
df["Test"] = df["Temp_Test"].apply(map_test)
df = df.drop(columns=["Temp_Test"])
if df["Test"].isna().any():
    logger.warning("Some Temp_Test values could not be mapped to Test.")

logger.info(f'Unique Tests: {df["Test"].unique()} ({df["Test"].nunique()})')

2025-08-27 17:56:33.167 | INFO     | __main__:<module>:10 - Unique patients before filtering: 31612
2025-08-27 17:57:12.994 | INFO     | __main__:<module>:22 - Unique patients after filtering patients < 3 years: 3329
2025-08-27 17:57:13.830 | WARNING  | __main__:<module>:39 - Some Temp_Test values could not be mapped to Test.
2025-08-27 17:57:13.882 | INFO     | __main__:<module>:41 - Unique Tests: ['BD' 'PFT' 'COD' None] (3)


In [225]:
none_rows = df[df["Test"].isna()]

In [226]:
none_rows

,Prescription Code,Patient Number,Gender,Date of Birth,Visit Type,Treatment Date,Prescription Name,Prescription Date,Implementation Date,Result item name,Result Numerical Value,Laboratory,Implementation laboratory,Region,Result Value,Pacs Number,Variable,Measurement,Test
320,FF6006,13127,M,19421226.0,O,20241112.0,Plethysmography,20241112,20250121.0,TLC_Pred,5.13,NaN,NaN,NaN,NaN,NaN,Pred,TLC,None
321,FF6006,13127,M,19421226.0,O,20241112.0,Plethysmography,20241112,20250121.0,TLC_Meas,4.95,NaN,NaN,NaN,NaN,NaN,Meas,TLC,None
322,FF6006,13127,M,19421226.0,O,20241112.0,Plethysmography,20241112,20250121.0,TLC_%Pred,97.0,NaN,NaN,NaN,NaN,NaN,%Pred,TLC,None
323,FF6006,13127,M,19421226.0,O,20241112.0,Plethysmography,20241112,20250121.0,VC_Pred,3.32,NaN,NaN,NaN,NaN,NaN,Pred,VC,None
324,FF6006,13127,M,19421226.0,O,20241112.0,Plethysmography,20241112,20250121.0,VC_Meas,3.28,NaN,NaN,NaN,NaN,NaN,Meas,VC,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3285510,OPF6006G,1148692,NaN,NaN,NaN,NaN,Plethysmography,20240223,NaN,Plethysmography ERV Meas,1.13,PU01,-,,1.13,2024022346039.0,Meas,ERV,None
3285511,OPF6006G,1148692,NaN,NaN,NaN,NaN,Plethysmography,20240223,NaN,Plethysmography ERV %Pred,132.0,PU01,-,,132,2024022346039.0,%Pred,ERV,None
3285512,OPF6006G,1148692,NaN,NaN,NaN,NaN,Plethysmography,20240223,NaN,Plethysmography RV Pred,2.0,PU01,-,,2,2024022346039.0,Pred,RV,None
3285513,OPF6006G,1148692,NaN,NaN,NaN,NaN,Plethysmography,20240223,NaN,Plethysmography RV Meas,2.54,PU01,-,,2.54,2024022346039.0,Meas,RV,None


In [227]:
df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))

C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_56832\3529406939.py:1: DtypeWarning: Columns (2,4,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))


In [228]:
uniq_test = df["Test"].unique()
uniq_mes = df["Measurement"].unique()
uniq_var = df["Variable"].unique()

In [229]:
print(f"{uniq_test=}")
print(f"{uniq_mes=}")
print(f"{uniq_var=}")

uniq_test=array(['BD', 'PFT', 'COD'], dtype=object)
uniq_mes=array(['FVC', 'FEV1', 'FEV1/FVC', 'DLCO'], dtype=object)
uniq_var=array(['Post_Meas', 'Post_%Pred', 'Post_%Chg', 'Pred', 'Meas', '%Pred',
       '%Chg.'], dtype=object)


In [3]:
import re

def plot_variable_statistics(df_path: str | Path = INTERIM_DATA_DIR / "ALL_PRESCRIPTION_DATA_FILTERED.csv",
                             output_dir: Path = Path(FIGURES_DIR / "Variant_statistics")
                             ) -> None:
    
    plt.rcParams.update({
    "font.size": 8,       # base font size
    "axes.titlesize": 10,  # title
    "axes.labelsize": 8,  # x and y labels
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8
    })
    # plt.rcParams.update(plt.rcParamsDefault)

    df = pd.read_csv(df_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output directory created at: {output_dir}")
    
    df["Prescription Date"] = pd.to_datetime(df["Prescription Date"], format="%Y%m%d", errors="coerce")
    df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")

    for test in df["Test"].unique():
        for meas in df["Measurement"].unique():
            for var in ['Meas', '%Pred', '%Chg.', 'Post_Meas', 'Post_%Chg', 'Post_%Pred']: #df["Variable"].unique():
                df_filtered = df[(df["Test"] == test) & (df["Measurement"] == meas) & (df["Variable"] == var)]
                if df_filtered.empty:
                    continue
                # Replace problematic characters (/ \ spaces) with underscore
                safe_test  = re.sub(r"[\\/]", "_", str(test))
                safe_meas = re.sub(r"[\\/]", "_", str(meas))
                safe_var  = re.sub(r"[\\/]", "_", str(var))
                print(f"Test: {test} , Measurement: {meas}, Variable: {var} | "
                      f"Unique: {df_filtered['Patient Number'].nunique()} | "
                      f"Min: {df_filtered['Result Numerical Value'].min():.2f}, "
                      f"Max: {df_filtered['Result Numerical Value'].max():.2f}, "
                      f"Mean: {df_filtered['Result Numerical Value'].mean():.2f}")
                # plot
                # plt.figure(figsize=(10, 6))
                plt.figure(figsize=(4.5, 2.7))
                plt.hist(df_filtered['Result Numerical Value'].dropna(), bins=30, color='skyblue', alpha=0.7)
                plt.xlabel('Result Numerical Value')
                plt.ylabel('Frequency')
                plt.title(
                    f"Distribution of '{var}' for '{test}' - '{meas}'\n"
                    f"Unique Patients: {df_filtered['Patient Number'].nunique()}\n"
                    f"Min: {df_filtered['Result Numerical Value'].min():.2f}, "
                    f"Max: {df_filtered['Result Numerical Value'].max():.2f}, "
                    f"Mean: {df_filtered['Result Numerical Value'].mean():.2f}"
                )
                plt.tight_layout()
                plt.savefig(Path(output_dir) / f"{safe_var}_{safe_test}_{safe_meas}_distribution.png")
                plt.close()

plot_variable_statistics()

C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_72040\2001071681.py:17: DtypeWarning: Columns (2,4,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df_path)


Output directory created at: D:\Research\Project_COPD\COPD\reports\figures\Variant_statistics
Test: BD , Measurement: FVC, Variable: Meas | Unique: 3086 | Min: 0.00, Max: 7.10, Mean: 3.17
Test: BD , Measurement: FVC, Variable: %Pred | Unique: 3086 | Min: -85.43, Max: 179.00, Mean: 93.10
Test: BD , Measurement: FVC, Variable: %Chg. | Unique: 3044 | Min: -11.62, Max: 57.00, Mean: 1.68
Test: BD , Measurement: FVC, Variable: Post_Meas | Unique: 1331 | Min: 0.90, Max: 6.77, Mean: 3.24
Test: BD , Measurement: FVC, Variable: Post_%Chg | Unique: 1331 | Min: -6.00, Max: 46.00, Mean: 1.87
Test: BD , Measurement: FVC, Variable: Post_%Pred | Unique: 1331 | Min: 29.00, Max: 157.00, Mean: 96.69
Test: BD , Measurement: FEV1, Variable: Meas | Unique: 3086 | Min: 0.00, Max: 6.02, Mean: 2.02
Test: BD , Measurement: FEV1, Variable: %Pred | Unique: 3086 | Min: -70.48, Max: 282.00, Mean: 84.89
Test: BD , Measurement: FEV1, Variable: %Chg. | Unique: 3044 | Min: -34.00, Max: 84.00, Mean: 5.66
Test: BD , Meas

In [56]:
wanted_measure_indexes = sorted(dfx[measure_col].dropna().unique().tolist())

In [57]:
wanted_measure_indexes

['DLCO', 'FEV1', 'FEV1/FVC', 'FVC']

In [58]:
base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}
measure_index_colors

{'DLCO': 'C0', 'FEV1': 'C1', 'FEV1/FVC': 'C2', 'FVC': 'C3'}